<a href="https://www.kaggle.com/code/secretiveplotter1863/task-2-part-a?scriptVersionId=335982890" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>


Part A: Classical Machine Learning - The Baseline
Objective: Use traditional machine learning to perform sentiment analysis on a text dataset (IMDb Reviews,
Twitter Sentiment, Amazon Reviews, or any suitable dataset), establishing the classical baselines every later
experiment is measured against.



In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

In [ ]:
path='/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv'
IMDB=pd.DataFrame(pd.read_csv(path))
IMDB.head()

In [ ]:
#inspect columns
print(IMDB.shape)
print(IMDB.columns)
print(IMDB["sentiment"].value_counts())
print(IMDB["sentiment"])


In [ ]:
IMDB["sentiment"] = IMDB["sentiment"].map({"positive": 1, "negative": 0})
print(IMDB["sentiment"])

In [ ]:
import re
def clean(text):
  text=text.lower()
  text=re.sub('<.*?>',' ',text)
  text=re.sub('[^a-z]+',' ',text)
  text=re.sub(r'\s+',' ',text).strip()
  return text

In [ ]:
IMDB["clean_review"]=IMDB["review"].apply(clean)
print(IMDB[["review","clean_review"]])

In [ ]:
x=IMDB["clean_review"]
y=IMDB["sentiment"]
#Train and test
from sklearn.model_selection import train_test_split
x_train,x_temp,y_train,y_temp=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

x_val,x_test,y_val,y_test=train_test_split(x_temp,y_temp,test_size=0.5,random_state=42,stratify=y_temp)

print("X Train shape: ",x_train.shape)
print("X val: ",x_val.shape)
print("x test: ",x_test.shape)

In [ ]:
print("Train class counts:\n", y_train.value_counts())
print("Val class counts:\n", y_val.value_counts())
print("Test class counts:\n", y_test.value_counts())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.95
)

In [ ]:
x_train_tfidf=tfidf.fit_transform(x_train)
x_val_tfidf=tfidf.transform(x_val)
x_test_tfidf=tfidf.transform(x_test)
#printing statements
print("Train TF-IDF shape: ",x_train_tfidf.shape)
print("val TF-IDF shape: ",x_val_tfidf.shape)
print("test TF-IDF shape: ",x_test_tfidf.shape)
print("Vocab size:", len(tfidf.vocabulary_))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score
logistic=LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)


In [ ]:
logistic.fit(x_train_tfidf, y_train)

y_val_pred_log = logistic.predict(x_val_tfidf)
print("Logistic Regression – Validation accuracy:",
      accuracy_score(y_val, y_val_pred_log))

print("\nLogistic Regression – Validation classification report:")
print(classification_report(y_val, y_val_pred_log))

In [ ]:
from sklearn.svm import LinearSVC

svm_clf = LinearSVC(
    C=1.0,
    random_state=42,
)

svm_clf.fit(x_train_tfidf, y_train)

y_val_pred_svm = svm_clf.predict(x_val_tfidf)
print("Linear SVM – Validation accuracy:",
      accuracy_score(y_val, y_val_pred_svm))

print("\nLinear SVM – Validation classification report:")
print(classification_report(y_val, y_val_pred_svm))

In [ ]:
# Logistic Regression on test
y_test_pred_log = logistic.predict(x_test_tfidf)
print("Logistic Regression – Test accuracy:",
      accuracy_score(y_test, y_test_pred_log))
print("\nLogistic Regression – Test classification report:")
print(classification_report(y_test, y_test_pred_log))

# SVM on test
y_test_pred_svm = svm_clf.predict(x_test_tfidf)
print("Linear SVM – Test accuracy:",
      accuracy_score(y_test, y_test_pred_svm))
print("\nLinear SVM – Test classification report:")
print(classification_report(y_test, y_test_pred_svm))

In [ ]:
# choose one model, e.g. SVM
import numpy as np

mis_idx = np.where(y_test != y_test_pred_svm)[0]
print("Number of misclassified test examples (SVM):", len(mis_idx))

# show first 5
for i in mis_idx[:5]:
    print("----")
    print("True label:", y_test.iloc[i])
    print("Predicted:", y_test_pred_svm[i])
    print("Review:", x_test.iloc[i][:500])  # first 500 chars
